In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y numpy pandas scipy scikit-learn librosa numba nemo-toolkit
!pip install -q numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.3.2 librosa==0.10.1 numba==0.59.1
!pip install -q "nemo-toolkit[asr,tts]==2.5.0"

In [ ]:
import numpy as np
import pandas as pd
import torch
import nemo.collections.asr as nemo_asr

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print("NeMo ASR works")

In [ ]:
# creating the ASR dataset
!mkdir -p data/librispeech
%cd /content
!wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz
!tar -xzf dev-clean.tar.gz -C /content/data/librispeech

In [ ]:
!find /content/data/librispeech/LibriSpeech/dev-clean -name "*.flac" | head

In [ ]:
# cretae manifest files for NeMo

import os
import json
import random
import soundfile as sf
from pathlib import Path

librispeech_root = Path("/content/data/librispeech/LibriSpeech/dev-clean")
output_dir = Path("/content/manifests")
output_dir.mkdir(parents=True, exist_ok=True)

items = []

for trans_file in librispeech_root.rglob("*.trans.txt"):
    with open(trans_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            utt_id = parts[0]
            text = parts[1].lower()

            audio_path = trans_file.parent / f"{utt_id}.flac"
            info = sf.info(str(audio_path))
            duration = info.duration

            items.append({
                "audio_filepath": str(audio_path),
                "duration": duration,
                "text": text
            })

print("Total samples:", len(items))
print(items[0])

#split dataset intro train test validation sets
random.seed(42)
random.shuffle(items)

n = len(items)
train = items[:int(0.8*n)]
val = items[int(0.8*n):int(0.9*n)]
test = items[int(0.9*n):]

def write_manifest(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

write_manifest(train, output_dir / "train_manifest.json")
write_manifest(val, output_dir / "val_manifest.json")
write_manifest(test, output_dir / "test_manifest.json")

print(len(train), len(val), len(test))

In [ ]:
# check files
!head -n 2 /content/manifests/train_manifest.json

In [ ]:
# baseline inference

import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(
    model_name="stt_en_citrinet_256"
)

# testing the baseline inference on 5 files
sample_audio = [item["audio_filepath"] for item in test[:5]]
sample_refs = [item["text"] for item in test[:5]]

preds = asr_model.transcribe(sample_audio)

for ref, pred in zip(sample_refs, preds):
    print("REF :", ref)
    print("PRED:", pred)
    print("-" * 80)

In [ ]:
# Baseline WER - pretrained model
asr_model.test_dataloader = None
trainer.test(asr_model, dataloaders=test_dl)

In [ ]:
# fine tuning

from omegaconf import OmegaConf

# manifests location
train_manifest = "/content/manifests/train_manifest.json"
val_manifest = "/content/manifests/val_manifest.json"

asr_model.setup_training_data(
    train_data_config={
        "manifest_filepath": train_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": True,
    }
)

asr_model.setup_validation_data(
    val_data_config={
        "manifest_filepath": val_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": False,
    }
)

from omegaconf import open_dict

with open_dict(asr_model.cfg.optim):
    asr_model.cfg.optim.lr = 1e-4

In [ ]:
# training
# pytorch dataloaders and defining batch size and shuffle behaviour

import lightning.pytorch as pl

trainer = pl.Trainer(
    # max_epochs=3,
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    log_every_n_steps=5
)

asr_model.set_trainer(trainer)
trainer.fit(asr_model)

In [ ]:
# evaluate WER on the test set
test_manifest = "/content/manifests/test_manifest.json"

asr_model.setup_test_data(
    test_data_config={
        "manifest_filepath": test_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": False,
    }
)

trainer.test(asr_model)

In [ ]:
predictions = []
references = []

asr_model.eval()
for batch in test_dl:
    signals, lengths, transcripts, transcript_lengths = batch
    with torch.no_grad():
        preds = asr_model.transcribe([...])
    predictions.extend(preds)
    references.extend(transcripts)

# Print 10 examples side by side
for ref, pred in zip(references[:10], predictions[:10]):
    print(f"REF:  {ref}")
    print(f"PRED: {pred}")
    print()

In [ ]:
# save model
asr_model.save_to("/content/citrinet_librispeech_finetuned.nemo")

In [ ]:
import json

test_items = []
with open("/content/manifests/test_manifest.json", "r", encoding="utf-8") as f:
    for line in f:
        test_items.append(json.loads(line))

subset = test_items[:20]
audio_files = [x["audio_filepath"] for x in subset]
references = [x["text"] for x in subset]

predictions = asr_model.transcribe(audio_files)

with open("/content/asr_test_predictions.txt", "w", encoding="utf-8") as f:
    for i, (ref, pred) in enumerate(zip(references, predictions), 1):
        pred_text = pred.text if hasattr(pred, "text") else str(pred)
        f.write(f"Example {i}\n")
        f.write(f"REF:  {ref}\n")
        f.write(f"PRED: {pred_text}\n")
        f.write("-" * 80 + "\n")

print("Saved to /content/asr_test_predictions.txt")